# 07. Выгрузка данных для Power BI (этап 8)

Готовим «чистые» таблицы для дашборда в папку `dashboard/data/`. В Power BI они загружаются как CSV
и связываются между собой по полям `district` и `listing_id`.

| Файл | Что внутри | Страница дашборда |
|---|---|---|
| `flats.csv` | все объявления (продажа и аренда) | Обзор рынка |
| `sale_investment.csv` | квартиры на продажу: доходность, окупаемость, справедливая цена, ссылка | Доходность, Шорт-лист |
| `districts.csv` | справочник районов: берег, порядок сортировки | для связей и фильтров |
| `history.csv` | индексы цен и аренды в Астане против инфляции, 2015–2025 | Инфляция и депозит |
| `roi_scenarios.csv` | ROI за 5 лет по сценариям и берегам + депозит | Инфляция и депозит |
| `hypotheses.csv` | результаты проверки гипотез | Гипотезы |

Перед запуском нужно выполнить ноутбуки 03–06.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

OUT = Path("dashboard/data")
OUT.mkdir(parents=True, exist_ok=True)

def save(df, name):
    # utf-8-sig — чтобы Power BI правильно прочитал кириллицу
    df.to_csv(OUT / name, index=False, encoding="utf-8-sig")
    print(f"{name:<22} {len(df):>5} строк, {df.shape[1]} колонок")

ROOMS_LABEL = {1: "1", 2: "2", 3: "3", 4: "4+"}
DEAL_LABEL = {"sale": "Продажа", "rent": "Аренда"}

## 1. Справочник районов

In [2]:
districts = pd.DataFrame({
    "district": ["Есильский", "Нура", "Сарайшык", "Алматинский", "Сарыарка", "Байконур", "Не указан"],
    "bank": ["Левый берег", "Левый берег", "Правый берег", "Правый берег", "Правый берег", "Правый берег", "Не указан"],
    "district_order": [1, 2, 3, 4, 5, 6, 7],
})
save(districts, "districts.csv")

districts.csv              7 строк, 3 колонок


## 2. Все объявления

In [3]:
import glob
raw = pd.read_csv(sorted(glob.glob("data/raw/listings_*.csv"))[-1])[["id", "url", "address"]].rename(columns={"id": "listing_id"})

flats = pd.read_pickle("data/clean_flats.pkl")
flats = flats.merge(raw, on="listing_id", how="left")
flats_out = pd.DataFrame({
    "listing_id": flats["listing_id"],
    "deal": flats["deal"].astype(str).map(DEAL_LABEL),
    "district": flats["district"].astype(str),
    "rooms": flats["rooms_grp"].astype(int).map(ROOMS_LABEL),
    "area_m2": flats["area_m2"].round(1),
    "floor": flats["floor"],
    "floors_total": flats["floors_total"],
    "floor_type": flats["floor_type"].astype(str),
    "building_height": flats["building_height"].astype(str),
    "price_kzt": flats["price_kzt"],
    "price_per_m2": flats["price_per_m2"].round(0),
    "address": flats["address"],
    "url": flats["url"],
})
save(flats_out, "flats.csv")

flats.csv               5580 строк, 13 колонок


## 3. Квартиры на продажу: инвестиционные метрики

In [4]:
inv = pd.read_pickle("data/investment_metrics.pkl")
fair = pd.read_pickle("data/sale_with_fair_price.pkl")[["listing_id", "fair_price", "discount"]]
inv = inv.merge(fair, on="listing_id", how="left").merge(raw, on="listing_id", how="left")

budget = pd.cut(inv["price_kzt"], [0, 20e6, 35e6, 50e6, np.inf],
                labels=["до 20 млн", "20–35 млн", "35–50 млн", "50+ млн"])
sale_out = pd.DataFrame({
    "listing_id": inv["listing_id"],
    "district": inv["district"].astype(str),
    "rooms": inv["rooms_grp"].astype(int).map(ROOMS_LABEL),
    "area_m2": inv["area_m2"].round(1),
    "floor_type": inv["floor_type"].astype(str),
    "building_height": inv["building_height"].astype(str),
    "price_kzt": inv["price_kzt"],
    "price_mln": inv["price_mln"],
    "price_per_m2": inv["price_per_m2"].round(0),
    "budget_band": budget.astype(str),
    "in_client_budget": inv["in_client_budget"],
    "est_rent_month": inv["est_rent"].round(-3),
    "gross_yield": inv["gross_yield"].round(4),
    "net_yield": inv["net_yield"].round(4),
    "real_net_yield": inv["real_net_yield"].round(4),
    "payback_years": inv["payback_years"].round(1),
    "fair_price": inv["fair_price"].round(-3),
    "discount_to_fair": inv["discount"].round(4),
    "is_undervalued": inv["discount"] >= 0.25,
    "address": inv["address"],
    "url": inv["url"],
})
save(sale_out, "sale_investment.csv")

sale_investment.csv     2878 строк, 21 колонок


## 4. История цен и ROI-сценарии

In [5]:
IDX_FILE = "data/stat/housing_indices.xlsx"

def city_series(path, sheet, city="Астана"):
    t = pd.read_excel(path, sheet_name=sheet, header=None)
    header_row = t.index[t.iloc[:, 1].astype(str).str.match(r"^20\d\d")][0]
    years = t.iloc[header_row, 1:]
    row = t[t.iloc[:, 0].astype(str).str.strip().str.rstrip("*") == city].iloc[0, 1:]
    out = pd.Series(pd.to_numeric(row.values, errors="coerce"), index=pd.to_numeric(years.values, errors="coerce"))
    return out[out.index.notna()].astype(float).rename(lambda y: int(y))

hist = pd.DataFrame({
    "Вторичка": city_series(IDX_FILE, "Перепродажа квартир"),
    "Новостройки": city_series(IDX_FILE, "Продажа нового жилья"),
    "Аренда": city_series(IDX_FILE, "Аренда квартир"),
}) / 100
CPI = pd.Series({2016: 8.5, 2017: 7.1, 2018: 5.3, 2019: 5.4, 2020: 7.5,
                 2021: 8.4, 2022: 20.3, 2023: 9.8, 2024: 8.6, 2025: 12.3}) / 100 + 1
hist = hist.loc[CPI.index]
hist["Инфляция"] = CPI
cum = hist.cumprod()
cum.loc[2015] = 1.0
cum = (cum.sort_index() * 100).round(1)
history = cum.reset_index(names="year").melt(id_vars="year", var_name="series", value_name="index_2015_100")
save(history, "history.csv")

roi_cols = [c for c in inv.columns if c.startswith("roi5 | ")]
roi = []
for col in roi_cols:
    scen = col.replace("roi5 | ", "")
    for bank, g in inv.groupby(inv["bank"].astype(str)):
        roi.append({"scenario": scen, "bank": bank,
                    "roi_nominal": round(g[col].median(), 4),
                    "roi_real": round(g[f"real_roi5 | {scen}"].median(), 4)})
DEPOSIT_RATE, INFLATION = 0.147, 0.1104
roi = pd.DataFrame(roi)
roi["deposit_nominal"] = round((1 + DEPOSIT_RATE) ** 5 - 1, 4)
roi["deposit_real"] = round((1 + DEPOSIT_RATE) ** 5 / (1 + INFLATION) ** 5 - 1, 4)
roi["scenario_order"] = roi["scenario"].map({s: i for i, s in enumerate(dict.fromkeys(roi["scenario"]), 1)})
save(roi, "roi_scenarios.csv")

history.csv               44 строк, 3 колонок
roi_scenarios.csv         10 строк, 7 колонок


## 5. Гипотезы

In [6]:
hyp = pd.read_csv("data/hypotheses_results.csv")
verdicts = {
    "H1": ("Однушки доходнее трёшек", "Подтверждена"),
    "H2": ("Правый берег доходнее левого", "Подтверждена"),
    "H3 (крайние вместе)": ("Крайние этажи дешевле на 5%+", "Подтверждена (на границе)"),
}
hyp = hyp[hyp["гипотеза"].isin(verdicts)].copy()
hyp_out = pd.DataFrame({
    "hypothesis": hyp["гипотеза"].str.split(" ").str[0],
    "statement": hyp["гипотеза"].map(lambda h: verdicts[h][0]),
    "verdict": hyp["гипотеза"].map(lambda h: verdicts[h][1]),
    "group_a": hyp["группа A"], "median_a": hyp["медиана A"].round(4),
    "group_b": hyp["группа B"], "median_b": hyp["медиана B"].round(4),
    "effect": hyp["разница медиан"].round(4),
    "ci_low": hyp["ДИ 95% низ"].round(4), "ci_high": hyp["ДИ 95% верх"].round(4),
    "p_value": hyp["p-value"],
})
save(hyp_out, "hypotheses.csv")

hypotheses.csv             3 строк, 11 колонок
